# Greedy Algorithms

A paradigm: build a solution by repeatedly taking the **locally best** choice and never reconsidering. When it works it's beautifully simple and fast (usually a sort + one pass) — but it **only works when the problem has the greedy-choice property**, and proving that is the real task. We'll see classic wins (activity selection, fractional knapsack) and an instructive failure (coin change, which needs DP).

**Contents**

1. **The signal** — and the catch
2. **Activity selection** / interval scheduling
3. **Fractional knapsack**
4. **When greedy fails** — and the exchange argument
5. **When to use & cheat-sheet**

## 1. The signal — and the catch

A **greedy** algorithm makes the choice that looks best *right now* and never backtracks. The appeal is speed and simplicity — typically **sort, then one pass**. The catch is correctness: greedy is right only when the problem has the **greedy-choice property** (a locally optimal choice belongs to *some* global optimum) plus **optimal substructure**.

That's a real condition, not a given. The hard part of a greedy solution is **proving** the greedy choice is safe — usually via an *exchange argument* (§4). When the property fails, greedy returns a wrong answer and you need **dynamic programming** instead (the DP notebook). Rule of thumb: *greedy commits and never reconsiders; DP explores all options.* Use greedy only when you can argue that committing is safe.

## 2. Activity selection / interval scheduling

Given intervals, select the **most** that don't overlap. The greedy choice: always take the interval that **finishes earliest** — it leaves the most room for the rest. Sort by end time, then sweep, taking each interval whose start is ≥ the last chosen end:

In [1]:
def max_activities(intervals):
    chosen, last_end = [], float("-inf")
    for start, end in sorted(intervals, key=lambda iv: iv[1]):   # earliest finish first
        if start >= last_end:
            chosen.append((start, end))
            last_end = end
    return chosen

print("selected:", max_activities([(1, 3), (2, 5), (4, 7), (1, 8), (5, 9), (8, 10)]))


selected: [(1, 3), (4, 7), (8, 10)]


"Earliest finish" is the provably-optimal greedy choice here; "earliest start" or "shortest interval" are tempting but **wrong**. Picking the right greedy criterion is the whole game — and this is the sweep-line family from the patterns tier.

## 3. Fractional knapsack

Maximize value in a capacity-limited knapsack when you can take **fractions** of items. The greedy choice: take items by **value-to-weight ratio**, filling completely until the last item, which you take partially. Sorting by ratio makes it O(n log n):

In [2]:
def fractional_knapsack(items, capacity):    # items: (value, weight)
    total = 0.0
    for value, weight in sorted(items, key=lambda iw: iw[0] / iw[1], reverse=True):
        take = min(weight, capacity)          # all of it, or whatever fits
        total += value * take / weight
        capacity -= take
        if capacity == 0:
            break
    return total

print("fractional_knapsack:", fractional_knapsack([(60, 10), (100, 20), (120, 30)], 50))


fractional_knapsack: 240.0


The **fractional** version is greedy-solvable; the **0/1** knapsack (each item all-or-nothing) is **not** — greedy by ratio can be arbitrarily wrong there, which is exactly why it needs DP (the DP notebook). One word — "fractional" — flips the right tool.

## 4. When greedy fails — and the exchange argument

The cautionary tale is **coin change**. With canonical coin systems (like US coins) greedy works — always take the largest coin that fits. For an arbitrary set it can be wrong:

In [3]:
def greedy_coins(coins, amount):
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1

print("greedy [1,5,10,25], 30:", greedy_coins([1, 5, 10, 25], 30), "(optimal)")
print("greedy [1,3,4], 6     :", greedy_coins([1, 3, 4], 6), "(WRONG: 4+1+1; DP finds 3+3 = 2)")


greedy [1,5,10,25], 30: 2 (optimal)
greedy [1,3,4], 6     : 3 (WRONG: 4+1+1; DP finds 3+3 = 2)


The `[1,3,4]` case (straight from the DP notebook) shows greedy committing to the 4 and missing 3+3. **Why does greedy work for activity selection but not this?** The proof tool is the **exchange argument**: show any optimal solution can be transformed step by step into the greedy one *without getting worse*, so greedy is at least as good. When you can make that argument (earliest-finish, ratio-sort), greedy is safe; when you can't, it isn't a greedy problem.

## 5. When to use & cheat-sheet

| Problem | Greedy choice | Works? |
|---|---|:---:|
| Activity selection | earliest finish time | ✅ provable |
| Fractional knapsack | highest value/weight ratio | ✅ |
| Huffman coding | merge the two least-frequent | ✅ |
| Dijkstra / Prim / Kruskal | nearest / cheapest edge | ✅ (graph-algorithms) |
| Jump game (reachability) | farthest reach so far | ✅ |
| 0/1 knapsack | by ratio | ❌ → DP |
| Coin change (arbitrary coins) | largest coin | ❌ → DP |

**The decision:** greedy if you can **prove** the greedy-choice property (exchange argument); otherwise **DP** explores the options greedy would wrongly discard. Many greedy algorithms are just a `sorted(...)` plus a one-pass scan — cheap when they're correct. One more, the **jump game** (greedily track the farthest reachable index):

In [4]:
def can_jump(nums):                          # can you reach the last index?
    reach = 0
    for i, n in enumerate(nums):
        if i > reach:                        # this index is unreachable
            return False
        reach = max(reach, i + n)            # greedily extend the farthest reach
    return True

print("can_jump([2,3,1,1,4]):", can_jump([2, 3, 1, 1, 4]))
print("can_jump([3,2,1,0,4]):", can_jump([3, 2, 1, 0, 4]))


can_jump([2,3,1,1,4]): True
can_jump([3,2,1,0,4]): False
